# Sudoku Part 2 — Indirect Solution #1

This notebook explains the variables, domains, predicates and axioms used in
**Indirect Solution #1** from Section 2.3 of the paper. We will start with 4x4 size Sudokus, comparing it to the pure CNN approach.

The key idea: instead of classifying the whole puzzle, the LTN learns to classify
each cell digit using **only puzzle-level validity labels** as supervision.
No digit labels are ever given, the sudoku rules guides the CNN training. I won't use the exact solution that the paper code proposes, we will just follow authors' idea.

---

## Working example

We use a single fixed 4×4 valid puzzle throughout:

```
Position  (i,j)    digit
─────────────────────────
(0,0) (0,1) (0,2) (0,3)    →   0  1  2  3   row 0
(1,0) (1,1) (1,2) (1,3)    →   2  3  0  1   row 1
(2,0) (2,1) (2,2) (2,3)    →   1  2  3  0   row 2
(3,0) (3,1) (3,2) (3,3)    →   3  0  1  2   row 3
```

This puzzle is **valid**: every row, column, and 2×2 block contains digits {0,1,2,3} exactly once.
Cell index in the flat list = `i * 4 + j`, so flat index 0 = `(0,0)`, index 5 = `(1,1)`, etc.

## LTN Architecture — Indirect Solution #1

![LTN Indirect Solution #1](/assets/jupyter/ltn_indirect1_diagram.svg)

## 1. Domains

Domains are the **types** of objects the logic can talk about.

| Domain | What it contains | 4×4 example |
|--------|-----------------|-------------|
| **images** | 28×28 pixel images, one per cell | 16 grayscale images |
| **digits** | The digit symbols | {0, 1, 2, 3} |
| **coordinates** | flat cell index in the puzzle | 0 … 15 |
| **sudoku** | A full puzzle (all 16 images together) | 1 puzzle at a time |
| **results** | Boolean puzzle validity | True (1) or False (0) |

Every variable and predicate input/output belongs to one of these domains.

## 2. Variables

A variable ranges over all individuals in a domain.
In LTN this becomes an `ltn.Variable` — a named tensor whose rows are the individuals.

---

### x: cell images
`x_{i,j}` is the 28×28 pixel image at row `i`, column `j`. Domain: **images**.

In code: `images_flat[i*n + j]` with shape `(28, 28, 1)`.
`x` is created **inside the training loop** because it changes every batch:
```python
images_flat = tf.reshape(puzzle_images, (-1, 28, 28, 1))  # (batch*16, 28, 28, 1)
x = ltn.Variable("x", images_flat)
```

---

### d: digit values
`d` ranges over `{0, 1, 2, 3}`. Domain: **digits**.

Defined **once outside the loop** — the set of digits never changes.
It is a `Variable` (not a `Constant`) because we quantify over it: `∀d`, `∃d`.
```python
d = ltn.Variable("d", tf.cast(tf.range(n_classes), tf.float32))
```

---

### se: sub-elements (plain Python list, not an LTN Variable)
`se_indices` is a plain Python list of 12 lists, each holding 4 flat cell indices.
Domain: **coordinates**.

| Sub-element | Flat indices |
|-------------|-------------|
| row 0 | [0, 1, 2, 3] |
| row 1 | [4, 5, 6, 7] |
| row 2 | [8, 9, 10, 11] |
| row 3 | [12, 13, 14, 15] |
| col 0 | [0, 4, 8, 12] |
| col 1 | [1, 5, 9, 13] |
| col 2 | [2, 6, 10, 14] |
| col 3 | [3, 7, 11, 15] |
| block (0,0) | [0, 1, 4, 5] |
| block (0,1) | [2, 3, 6, 7] |
| block (1,0) | [8, 9, 12, 13] |
| block (1,1) | [10, 11, 14, 15] |

We **cannot** use `ltn.Variable("se", ...)` here because LTN's `Forall(se, formula)`
requires `se` to be a *free variable* inside `formula`.
`valid_element` depends on `x` and `d`, not on an index variable — so LTN cannot
aggregate over `se` automatically.
Instead, `valid_element_sat` iterates over `se_indices` in Python and gathers each
sub-element's cell images with `tf.gather`.

Defined **once outside the loop**:
```python
se_indices = get_sub_elements(n_classes)   # plain list, shape (12, 4)
```

---

### l: labels
`l` ranges over the validity labels for each puzzle in the batch: 0.0 (invalid) or 1.0 (valid).
Domain: **results**.

Defined **inside `compute_axioms`** (batch changes every step):
```python
l = ltn.Variable("l", tf.cast(puzzle_labels, tf.float32))  # (batch,)
```

Each axiom uses `mask=` to filter `l` to only the relevant subset:
Axiom 2 sees only `l=1.0` (valid puzzles), Axiom 3 sees only `l=0.0` (invalid puzzles).
The `mask` is a `Formula` object with the same free variables as the quantified formula.

## 3. Predicates

Predicates map individuals to a truth value in **[0, 1]**.

---

### digit(x, d) — digit classifier

Returns the probability that cell image `x` contains digit `d`.

- **Input:** one cell image `(28,28,1)` + one digit value
- **Output:** probability in [0, 1]
- **Grounded by:** `DigitCNN` — the only trainable component
- **Not** the Part 1 CNN: this CNN is trained from scratch via the LTN loss,
  with no digit-level supervision. The LTN axioms are the only training signal.

Example for cell (0,0) whose true digit is 0:
```
digit(x_{0,0}, 0) ≈ 0.97   ← CNN is confident it's a 0
digit(x_{0,0}, 1) ≈ 0.01
digit(x_{0,0}, 2) ≈ 0.01
digit(x_{0,0}, 3) ≈ 0.01
```

`ltn.Predicate` requires a `tf.keras.Model`. `DigitGrounding` wraps `DigitCNN` and adds
the `d`-indexing logic, keeping `DigitCNN` reusable as a plain classifier in other notebooks.

When LTN evaluates `Digit([x, d])`, it broadcasts `x` (N_images) and `d` (4 digits) into
N = N_images × 4 pairs. Inside `call`, both `images` and `d` are already flattened to length N,
so we must select `probs[i, idx[i]]` per sample using `tf.gather_nd`.

```python
class DigitGrounding(tf.keras.Model):
    def call(self, inputs, training=False):
        images, d = inputs
        probs = cnn(images, training=training)               # (N, n_classes)
        idx = tf.cast(tf.squeeze(d, axis=-1), tf.int32)      # (N,)
        row_idx = tf.range(tf.shape(probs)[0])
        return tf.gather_nd(probs, tf.stack([row_idx, idx], axis=1))  # (N,)

Digit = ltn.Predicate(DigitGrounding())
```

---

### validElement(se) — sub-element validity

`validElement(se)` is **not a separate predicate object** — it is a derived LTN formula:

```
validElement(se)  ≡  ∀d  ∃x : digit(x, d)
```

"For every digit d, there is some cell whose CNN output for d is high."

In code:
```python
valid_element = Forall(d, Exists(x, Digit([x, d])))
```

---

### The LTN execution model: Term → Formula → scalar

This is a key concept that explains how all LTN notebooks work.

**Every `ltn.Predicate.__call__` returns a `Formula` object** — never a plain tensor,
never a Variable. The `Formula` carries two things:

| attribute | meaning |
|-----------|---------|
| `.tensor` | the truth-value tensor |
| `.free_vars` | list of variable labels that index the tensor's axes |

The standard flow is always:

```
Variable (Term) ──[Predicate]──► Formula   (free_vars populated)
Formula + Formula ──[Connective]──► Formula
Formula ──[Quantifier]──► Formula          (one free_var removed per quantifier)
```

You **cannot** pass a `Variable` directly to a connective — `Wrapper_Connective`
checks `isinstance(x, Formula)` and raises `TypeError` otherwise. The Predicate
is the only crossing point between Terms and Formulas.

**Concrete example** — `Forall(d, Exists(x, Digit([x, d])))`:

```
Digit([x, d])           → Formula   tensor shape (N_x × N_d,)   free_vars=["x", "d"]
Exists(x, ...)          → Formula   tensor shape (N_d,)          free_vars=["d"]
Forall(d, ...)          → Formula   tensor shape ()              free_vars=[]
```

Each quantifier finds its variable in `free_vars`, aggregates along that axis,
and removes it from the list. If the variable is not in `free_vars`, LTN raises
`ValueError: 'x' is not in list`.

**What about `l`?** — puzzle labels are already truth values (0.0/1.0), so there
is no predicate to apply. To use `l` in connectives and quantifiers we must cross
the Term → Formula boundary manually:

```python
l_f = Formula(l.tensor, free_vars=["l"])   # same data, typed as Formula
```

This is exactly what a Predicate would do internally — we just skip the model call
because the tensor is already the truth value we want.

## 4. The Two Axioms of Indirect Solution #1

From Section 2.3, Equations (2)–(3) of the paper.
Axiom 1 in the paper is definitional (it names `validElement`) — since we define
`valid_element` as a Python variable inline, we do not need it as a training axiom.

---

### Computing validElement per sub-element

Because `se` cannot be an LTN Variable (see Variables section), we iterate over the
12 sub-elements in Python, gather each one's cell images, and compute its satisfaction:

```python
def valid_element_sat(images_flat, se_indices, training):
    se_sats = []
    for cell_indices in se_indices:          # 12 sub-elements
        x_se = ltn.Variable("x_se", tf.gather(images_flat, cell_indices))
        ve   = Forall(d, Exists(x_se, Digit([x_se, d], training=training)))
        se_sats.append(ve.tensor)            # scalar per sub-element
    return tf.stack(se_sats)                 # (12,)
```

`tf.gather(images_flat, cell_indices)` picks the 4 images belonging to this sub-element
from the flat pool of all batch images.

---

### Building Formula objects for the axioms

`valid_element_sat` returns a plain `(12,)` tensor — not a `Formula`.
We aggregate it directly with the same fuzzy operations that `Forall`/`Exists` use internally,
then broadcast and wrap the results as `Formula` objects so the `l`-level quantifiers work.

**Step 1 — aggregate over sub-elements**:

```python
all_valid  = ltn.fuzzy_ops.Aggreg_pMeanError(p=2)(se_sat, axis=0)  # scalar  (mirrors ∀se)
exists_inv = ltn.fuzzy_ops.Aggreg_pMean(p=6)(1. - se_sat, axis=0) # scalar  (mirrors ∃se ¬ve)
```

These are the same aggregation functions that `Forall` and `Exists` wrap — applied directly
here because `se_sat` is a plain tensor, not an LTN Formula.

**Step 2 — create the `l` variable and wrap everything as Formulas**:

```python
l = ltn.Variable("l", labels)            # Variable (Term), shape (B,)

l_f     = Formula(l.tensor, free_vars=["l"])
not_l_f = Not(l_f)                        # Not accepts a Formula and returns one
ve_f    = Formula(tf.broadcast_to(all_valid,  tf.shape(labels)), free_vars=["l"])
inv_f   = Formula(tf.broadcast_to(exists_inv, tf.shape(labels)), free_vars=["l"])
```

`all_valid` is a scalar with no batch dimension. Broadcasting it to `(B,)` and declaring
`free_vars=["l"]` tells LTN "this value applies to every puzzle in `l`", so
`Forall(l, ...)` can find `"l"` in `free_vars` and aggregate over it.

---

### Axiom 2 — Valid puzzles

```
∀l=1  (l  ↔  (∀se : validElement(se)))
```

Applied only to valid puzzles (`l=1`): all sub-elements must be valid.

```python
mask_valid = Formula(tf.cast(l.tensor >= 0.5, tf.float32), free_vars=["l"])
ax2 = Forall(l, Equiv(l_f, ve_f), mask=mask_valid)
```

- `Equiv(l_f, ve_f)` → `Formula` with `free_vars=["l"]`, shape `(B,)`
- `Forall(l, ..., mask=mask_valid)` → aggregates over valid puzzles only → scalar

---

### Axiom 3 — Invalid puzzles

```
∀l=0  (¬l  ↔  (∃se : ¬validElement(se)))
```

Applied only to invalid puzzles (`l=0`): at least one sub-element must be invalid.

```python
mask_invalid = Formula(tf.cast(l.tensor < 0.5, tf.float32), free_vars=["l"])
ax3 = Forall(l, Equiv(not_l_f, inv_f), mask=mask_invalid)
```

For `l=0`: `not_l_f = Not(l_f) = 1.0`. `Equiv(1.0, exists_inv)` is high when `exists_inv` is high.

---

### Why `mask=` instead of `cond_fn=`?

The actual `Wrapper_Quantifier` API uses `mask=` (a `Formula` object), not `cond_fn=`.
The mask is cast to `bool` internally; entries where `mask=False` are skipped and the
quantifier's empty-set semantics applies (1.0 for `Forall`).

---

### Why two axioms, not one?

| | Axiom 2 (`∀se`) | Axiom 3 (`∃se`) |
|---|---|---|
| Valid puzzles (l=1) | strong signal — all se must pass | — |
| Invalid puzzles (l=0) | weak signal — `∀se` hard to push to 0 | strong signal — just one bad se needed |

Using `∀se` for invalid puzzles would require every sub-element to fail, which is too strong.
Using `∃se` for valid puzzles would allow trivially satisfying the axiom with one good sub-element.
Each quantifier is matched to the case where it gives the clearest gradient signal.

---

### Full chain

```
x_{i,j}  →  DigitGrounding  →  Digit([x,d])   Formula  free_vars=["x","d"]
                                      ↓
                  Exists(x_se, ...)             Formula  free_vars=["d"]
                  Forall(d,    ...)             scalar   free_vars=[]
                  [× 12 sub-elements, Python loop]
                                      ↓
                  valid_element_sat → (12,) tensor
                  Aggreg_pMeanError → all_valid  scalar   (∀se)
                  Aggreg_pMean     → exists_inv scalar   (∃se ¬ve)
                                      ↓
                  broadcast + Formula(…, ["l"])  → (B,) Formulas
                  Equiv(l_f, ve_f)              Formula  free_vars=["l"]
                  Forall(l, …, mask=…)          scalar   free_vars=[]
                                      ↓
                  formula_aggregator([ax2, ax3])
                  loss = 1 − sat  →  ∂loss/∂θ_CNN
```

## 5. Summary

| Symbol | LTN type | Defined | Meaning |
|--------|----------|---------|---------|
| `x` | `ltn.Variable` | inside loop | Cell images — changes every batch |
| `d` | `ltn.Variable` | outside loop | Digit values {0,1,2,3} — quantified over with `∀d`, `∃d` |
| `se_indices` | Python list | outside loop | Sub-element cell indices — iterated explicitly, not an LTN Variable |
| `l` | `ltn.Variable` | inside `compute_axioms` | Puzzle validity labels (0.0/1.0) — changes every batch |
| `Digit` | `ltn.Predicate` | outside loop | Wraps `DigitGrounding` — only trainable component |
| `valid_element_sat` | Python function | — | Returns `(12,)` tensor: `Forall(d, Exists(x_se, Digit([x_se,d])))` per se |

**Key distinctions:**
- `d` is a `Variable` not a `Constant` because we quantify over it (`∀d`, `∃d`)
- `se` is NOT an `ltn.Variable` — LTN's `Forall(se, formula)` requires `se` to be a free variable in `formula`, but `valid_element` depends on `x` and `d`, not on an index
- `Digit` is backed by `DigitGrounding(tf.keras.Model)` which wraps `DigitCNN`. It is needed because `ltn.Predicate` requires a Keras model, not a plain function
- Axiom 2 filters to `l=1.0` via `mask=mask_valid`. Axiom 3 filters to `l=0.0` via `mask=mask_invalid`. This is because the real LTN API uses `mask=` (a `Formula`), not `cond_fn=`
- The CNN is trained from scratch via `loss = 1 − formula_aggregator([ax2, ax3])`

In [ ]:
import sys, os
sys.path.insert(0, r"C:\MGEP Dropbox\Aitor Aguirre Ortuzar\Ikerketa\IAExplicable"
                   r"\NeuroSymbolicAI\LTN\logictensornetworks-master"
                   r"\logictensornetworks-master\learn\sudoku")
os.chdir(r"C:\MGEP Dropbox\Aitor Aguirre Ortuzar\Ikerketa\IAExplicable"
         r"\NeuroSymbolicAI\LTN\logictensornetworks-master"
         r"\logictensornetworks-master\learn\sudoku")

import ltn
from ltn.core import Formula
import tensorflow as tf
from DigitCNN import DigitCNN

# ── Sub-element index builder ──────────────────────────────────────────────
def get_sub_elements(n):
    b = int(n ** 0.5)
    se_list = []
    for i in range(n):
        se_list.append([i*n + j for j in range(n)])              # row i
    for j in range(n):
        se_list.append([i*n + j for i in range(n)])              # col j
    for br in range(b):
        for bc in range(b):
            se_list.append([
                (br*b + r)*n + (bc*b + c)
                for r in range(b) for c in range(b)
            ])                                                    # block (br,bc)
    return se_list

n_classes  = 4
se_indices = get_sub_elements(n_classes)   # plain list — NOT an ltn.Variable

# ── LTN grounding ──────────────────────────────────────────────────────────
cnn = DigitCNN(n_classes=n_classes)   # fresh CNN — trained via LTN loss only

class DigitGrounding(tf.keras.Model):
    def call(self, inputs, training=False):
        images, d = inputs
        probs = cnn(images, training=training)        # (N, n_classes)
        idx = tf.cast(tf.squeeze(d, axis=-1), tf.int32)   # (N,)
        # LTN broadcasts [x, d] into N=(n_images * n_digits) pairs.
        # Must select probs[i, idx[i]] for each i — tf.gather_nd, not tf.gather.
        row_idx = tf.range(tf.shape(probs)[0])
        return tf.gather_nd(probs, tf.stack([row_idx, idx], axis=1))  # (N,)

Digit = ltn.Predicate(DigitGrounding())

# ── Variables defined once (fixed across batches) ─────────────────────────
d = ltn.Variable("d", tf.cast(tf.range(n_classes), tf.float32))   # {0,1,2,3}
# x and l are defined inside compute_axioms (change every batch)

# ── Connectives and quantifiers ───────────────────────────────────────────
Not    = ltn.Wrapper_Connective(ltn.fuzzy_ops.Not_Std())
Equiv  = ltn.Wrapper_Connective(ltn.fuzzy_ops.Equiv(
             ltn.fuzzy_ops.And_Prod(),
             ltn.fuzzy_ops.Implies_Reichenbach()))
Forall = ltn.Wrapper_Quantifier(ltn.fuzzy_ops.Aggreg_pMeanError(p=2), semantics="forall")
Exists = ltn.Wrapper_Quantifier(ltn.fuzzy_ops.Aggreg_pMean(p=6),      semantics="exists")
formula_aggregator = ltn.Wrapper_Formula_Aggregator(ltn.fuzzy_ops.Aggreg_pMeanError(p=2))

optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

print("Setup complete.")
print(f"d          = {d.tensor.numpy().flatten()}")
print(f"se_indices = {len(se_indices)} sub-elements, {len(se_indices[0])} cells each")

In [ ]:


# ── valid_element_sat ─────────────────────────────────────────────────────
# Iterates over the 12 sub-elements in Python.
# For each, gathers its 4 cells from images_flat and computes ∀d ∃x: Digit(x,d).
# Returns a (12,) tensor — one satisfaction score per sub-element.

def valid_element_sat(images_flat, se_indices, training):
    se_sats = []
    for cell_indices in se_indices:
        x_se = ltn.Variable("x_se", tf.gather(images_flat, cell_indices))
        ve   = Forall(d, Exists(x_se, Digit([x_se, d], training=training)))
        se_sats.append(ve.tensor)
    return tf.stack(se_sats)   # (12,)


# ── compute_axioms ────────────────────────────────────────────────────────
# Called once per training step with the current batch.
# l is created fresh each call from the batch labels.

def compute_axioms(images_flat, puzzle_labels, se_indices, training=False):
    se_sat = valid_element_sat(images_flat, se_indices, training)  # (12,)

    # Aggregate over sub-elements — pMeanError mirrors ∀se, pMean mirrors ∃se
    all_valid  = ltn.fuzzy_ops.Aggreg_pMeanError(p=2)(se_sat, axis=0)  # scalar
    exists_inv = ltn.fuzzy_ops.Aggreg_pMean(p=6)(1. - se_sat, axis=0) # scalar

    labels = tf.cast(puzzle_labels, tf.float32)   # (B,)
    l = ltn.Variable("l", labels)  # needed for Forall — but Variable adds feature dim: l.tensor is (B,1)

    # Use labels (1D) not l.tensor (2D) — Variable.__init__ does tensor[:,tf.newaxis],
    # so l.tensor is (B,1). transpose_free_vars builds perm=[0] (1 free var), but
    # tf.transpose on a rank-2 tensor needs perm of size 2 → InvalidArgumentError.
    l_f     = Formula(labels, free_vars=["l"])
    not_l_f = Not(l_f)
    ve_f    = Formula(tf.broadcast_to(all_valid,  tf.shape(labels)), free_vars=["l"])
    inv_f   = Formula(tf.broadcast_to(exists_inv, tf.shape(labels)), free_vars=["l"])

    # Masks: select valid (l=1) or invalid (l=0) puzzles only
    mask_valid   = Formula(tf.cast(labels >= 0.5, tf.float32), free_vars=["l"])
    mask_invalid = Formula(tf.cast(labels <  0.5, tf.float32), free_vars=["l"])

    # Axiom 2: for valid puzzles (l=1), all sub-elements must be valid
    ax2 = Forall(l, Equiv(l_f, ve_f),      mask=mask_valid)
    # Axiom 3: for invalid puzzles (l=0), at least one sub-element must be invalid
    ax3 = Forall(l, Equiv(not_l_f, inv_f), mask=mask_invalid)

    return formula_aggregator([ax2, ax3]).tensor

print("Axioms defined.")

In [ ]:
from DataLoader import DataLoader

loader = DataLoader(dataset_name="mnist4", split=0, batch_size=32)
train_ds, val_ds, test_ds, n_classes = loader.load_dataset()

# NOTE: we loop over individual puzzles rather than batches.
# valid_element_sat uses flat cell indices (0–15) that are per-puzzle,
# so passing the whole batch's images_flat would always pick puzzle 0's cells.
# The production script (train_indirect1.py) uses vectorized batching for speed.
n_epochs = 500

for epoch in range(n_epochs):
    for puzzle_images, cell_labels, puzzle_labels in train_ds:
        B = puzzle_images.shape[0]
        for i in range(B):
            images_i = tf.reshape(puzzle_images[i], (-1, 28, 28, 1))  # (16, 28, 28, 1)
            label_i  = puzzle_labels[i:i+1]                            # (1,)

            with tf.GradientTape() as tape:
                sat  = compute_axioms(images_i, label_i, se_indices, training=True)
                loss = 1. - sat

            grads = tape.gradient(loss, cnn.trainable_variables)
            optimizer.apply_gradients(zip(grads, cnn.trainable_variables))

    if epoch % 100 == 0:
        val_sat_total, val_steps = 0., 0
        for puzzle_images, cell_labels, puzzle_labels in val_ds:
            B = puzzle_images.shape[0]
            for i in range(B):
                images_i = tf.reshape(puzzle_images[i], (-1, 28, 28, 1))
                label_i  = puzzle_labels[i:i+1]
                val_sat_total += compute_axioms(
                    images_i, label_i, se_indices, training=False
                ).numpy()
                val_steps += 1
        print("Epoch %d: Val Sat Level %.3f" % (epoch, val_sat_total / val_steps))

print("Training finished at Epoch %d" % epoch)

---

# Indirect Solution #2

Now we will implement the second indirect method of the paper.

Instead of the `validElement` axiom (∀d ∃x∈se: Digit(x,d)), Indirect Solution #2 uses a **pairwise constraint** directly on cell pairs:

```
∀(p1, p2) : Implies(SameSubElement(p1, p2), Not(Equal(p1, p2)))
```

"For every pair of cells that share a sub-element (row, col, or box), their predicted digits must differ."

The two axioms are:

| | Axiom |
|---|---|
| Valid puzzle (l=1) | `l → ∀(p1,p2): Implies(SameSubElement(p1,p2), Not(Equal(p1,p2)))` |
| Invalid puzzle (l=0) | `¬l → ∃(p1,p2): And(SameSubElement(p1,p2), Equal(p1,p2))` |

### Key differences from Indirect Solution #1

| | Indirect #1 | Indirect #2                                                        |
|---|---|--------------------------------------------------------------------|
| Inner axiom | `∀d ∃x∈se: Digit(x,d)`  uses digit labels | `∀(p1,p2): Implies(SameSubElement, Not(Equal))`  purely structural |
| Predicate | `Digit(x, d)`  learned, CNN-based | `Equal(p1,p2)` — learned; `SameSubElement`  fixed lookup           |
| Variables | `x` (images), `d` (digits) | `p1`, `p2` (cell indices 0..15)                                    |
| Sub-elements | iterated in Python (12 loops) | encoded in `same_se_mat` (16×16 matrix for our 4x4 Sudoku case)    |

### `Equal(p1, p2)`  fuzzy equality predicate. The paper defines it in this manner, but we could replace it with another distance metric of our choice:

```
Equal(p1, p2) = exp(−ReLU(‖CNN(p1) − CNN(p2)‖₁ − c))
```

- High (≈1) when both cells are predicted as the same digit
- Near 0 when they differ
- `c = 0.1` is a margin: pairs with L1 distance < c are treated as fully equal

### `SameSubElement(p1, p2)`  structural predicate

Returns 1 if cells p1 and p2 share any row, column, or box; 0 otherwise.
Backed by a precomputed (16×16) matrix — no learning involved.
Since it has no trainable parameters, it uses `ltn.Predicate.Lambda` instead of a full Keras model.

In [ ]:
import numpy as np
from ltn.core import Formula

# ── Structural constant: same_se_mat[i,j] = 1 if cells i and j share a sub-element ──
def get_same_se_matrix(n):
    n_cells = n * n
    mat = np.zeros((n_cells, n_cells), dtype=np.float32)
    for se in get_sub_elements(n):
        for i in se:
            for j in se:
                if i != j:
                    mat[i, j] = 1.0
    return tf.constant(mat)  # (n_cells, n_cells)

n_cells     = n_classes * n_classes
c           = 0.1                            # equality margin
same_se_mat = get_same_se_matrix(n_classes)  # (16, 16) — built once, never changes

# ── Fresh CNN for Indirect Solution #2 ───────────────────────────────────
cnn2 = DigitCNN(n_classes=n_classes)

# ── SameSubElement — parameter-free predicate, uses Predicate.Lambda ─────
# p1, p2 are cell indices (0..15); LTN broadcasts to all 256 pairs.
# ltn.Predicate.Lambda is used because there are no trainable parameters.
SameSubElement = ltn.Predicate.Lambda(
    lambda args: tf.gather_nd(
        same_se_mat,
        tf.cast(tf.stack([tf.squeeze(args[0], axis=-1),
                          tf.squeeze(args[1], axis=-1)], axis=1), tf.int32)
    )
)

# ── Equal — learned predicate (CNN-based) ────────────────────────────────
# _current_probs is set at the start of each compute_axioms call.
# EqualGrounding reads it via tf.gather — no second CNN inference.
_current_probs2 = None

class EqualGrounding(tf.keras.Model):
    def call(self, inputs, training=False):
        i = tf.cast(tf.squeeze(inputs[0], axis=-1), tf.int32)  # (N,)
        j = tf.cast(tf.squeeze(inputs[1], axis=-1), tf.int32)  # (N,)
        p1_probs = tf.gather(_current_probs2, i)               # (N, n_classes)
        p2_probs = tf.gather(_current_probs2, j)               # (N, n_classes)
        dist = tf.reduce_sum(tf.abs(p1_probs - p2_probs), axis=-1)  # (N,)
        return tf.exp(-tf.nn.relu(dist - c))                   # (N,)

Equal = ltn.Predicate(EqualGrounding())

# ── Variables: p1 and p2 range over cell indices 0..15 ───────────────────
# Defined once — the index set never changes across puzzles.
p1 = ltn.Variable("p1", tf.cast(tf.range(n_cells), tf.float32))
p2 = ltn.Variable("p2", tf.cast(tf.range(n_cells), tf.float32))

# ── Connectives and quantifiers ───────────────────────────────────────────
Not2    = ltn.Wrapper_Connective(ltn.fuzzy_ops.Not_Std())
And2    = ltn.Wrapper_Connective(ltn.fuzzy_ops.And_Prod())
Implies2 = ltn.Wrapper_Connective(ltn.fuzzy_ops.Implies_Reichenbach())
Exists2  = ltn.Wrapper_Quantifier(ltn.fuzzy_ops.Aggreg_pMean(p=6),      semantics="exists")
Forall2  = ltn.Wrapper_Quantifier(ltn.fuzzy_ops.Aggreg_pMeanError(p=6), semantics="forall")
formula_aggregator2 = ltn.Wrapper_Formula_Aggregator(ltn.fuzzy_ops.Aggreg_pMeanError(p=2))

optimizer2 = tf.keras.optimizers.Adam(learning_rate=0.0005)

print("Setup complete.")
print(f"same_se_mat shape : {same_se_mat.shape}")
print(f"p1 individuals    : {p1.tensor.numpy().flatten()}")

In [ ]:
def compute_axioms2(images_flat, puzzle_labels, training=False):
    global _current_probs2
    _current_probs2 = cnn2(images_flat, training=training)  # (n_cells, n_classes)

    labels = tf.cast(puzzle_labels, tf.float32)
    l      = ltn.Variable("l", labels)

    # Inner axioms — LTN broadcasts p1 × p2 into all 256 cell pairs
    ax_pairs_valid = Forall2([p1, p2], Implies2(SameSubElement([p1, p2]),
                                                Not2(Equal([p1, p2]))))
    ax_pairs_inv   = Exists2([p1, p2], And2(SameSubElement([p1, p2]),
                                            Equal([p1, p2])))

    ve_f  = Formula(tf.broadcast_to(ax_pairs_valid.tensor, tf.shape(labels)), free_vars=["l"])
    inv_f = Formula(tf.broadcast_to(ax_pairs_inv.tensor,   tf.shape(labels)), free_vars=["l"])

    mask_valid   = Formula(tf.cast(labels >= 0.5, tf.float32), free_vars=["l"])
    mask_invalid = Formula(tf.cast(labels <  0.5, tf.float32), free_vars=["l"])

    ax1 = Forall2(l, ve_f,  mask=mask_valid)    # valid puzzle  → all pairs differ
    ax2 = Forall2(l, inv_f, mask=mask_invalid)  # invalid puzzle → some pair collides

    return formula_aggregator2([ax1, ax2]).tensor

print("compute_axioms2 defined.")

In [ ]:
n_epochs2 = 500

for epoch in range(n_epochs2):
    for puzzle_images, cell_labels, puzzle_labels in train_ds:
        B = puzzle_images.shape[0]
        for i in range(B):
            images_i = puzzle_images[i]                       # (n_cells, 28, 28, 1)
            label_i  = tf.reshape(puzzle_labels[i], (1,))     # (1,)

            with tf.GradientTape() as tape:
                sat  = compute_axioms2(images_i, label_i, training=True)
                loss = 1. - sat

            grads = tape.gradient(loss, cnn2.trainable_variables)
            optimizer2.apply_gradients(zip(grads, cnn2.trainable_variables))

    if epoch % 100 == 0:
        val_sat_total, val_steps = 0., 0
        for puzzle_images, cell_labels, puzzle_labels in val_ds:
            B = puzzle_images.shape[0]
            for i in range(B):
                images_i = puzzle_images[i]
                label_i  = tf.reshape(puzzle_labels[i], (1,))
                val_sat_total += compute_axioms2(images_i, label_i).numpy()
                val_steps += 1
        print("Epoch %d: Val Sat Level %.3f" % (epoch, val_sat_total / val_steps))

print("Training finished at Epoch %d" % epoch)